# 第五章：Retrieval / RAG（检索增强生成）

## 学习目标

- 理解 RAG 解决什么问题（模型幻觉、私有数据）
- 掌握文档加载（Document Loaders）
- 掌握文本分割（Text Splitters）
- 理解 Embedding 向量化原理
- 使用向量数据库存储和检索（FAISS）
- 构建完整的 RAG 问答链

## 0. 环境准备

本章需要额外安装向量数据库依赖：

In [1]:
# 本章需要额外安装：
!uv add faiss-cpu

Resolved 64 packages in 6.42s                                        
⠦ Preparing packages... (0/1)                                                   ⠋ Preparing packages... (0/0)                                                   
⠧ Preparing packages... (0/1)-------------------     0 B/22.74 MiB           
⠧ Preparing packages... (0/1)-------------------     0 B/22.74 MiB           
⠇ Preparing packages... (0/1)------------------- 16.00 KiB/22.74 MiB         
⠇ Preparing packages... (0/1)------------------- 16.00 KiB/22.74 MiB         
⠇ Preparing packages... (0/1)------------------- 32.00 KiB/22.74 MiB         
⠇ Preparing packages... (0/1)------------------- 48.00 KiB/22.74 MiB         
⠋ Preparing packages... (0/1)------------------- 64.00 KiB/22.74 MiB         
⠋ Preparing packages... (0/1)------------------- 64.00 KiB/22.74 MiB         
⠋ Preparing packages... (0/1)------------------- 72.93 KiB/22.74 MiB         
⠋ Preparing packages... (0/1)------------------- 88.93 KiB/22.74 Mi

In [2]:
import os
from dotenv import load_dotenv
load_dotenv("../.env")
import langchain
print(f"langchain 版本: {langchain.__version__}")

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="kimi-k2.5",
    openai_api_base=os.environ["Qwen_API_BASE"],
    openai_api_key=os.environ["Qwen_API_KEY"],
    temperature=0.7,
)
print("模型初始化完成")

langchain 版本: 1.2.13
模型初始化完成


In [4]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(
    model="Embedding-3",
    openai_api_base=os.environ["GLM_API_BASE"],
    openai_api_key=os.environ["GLM_API_KEY"],
)
print("Embedding 模型初始化完成")

KeyError: 'GLM_API_BASE'

## 1. 对比：纯 LLM vs RAG（为什么需要 RAG）

先问模型一个它不可能知道的问题——关于我们自己的学习笔记内容：

In [4]:
# 直接问 LLM 一个关于私有数据的问题
response = llm.invoke("LangChain 学习项目的第三章讲了什么内容？")
print(response.content)

我无法直接回答这个问题，因为您没有提供具体的 LangChain 学习项目资料来源。

"第三章"的内容取决于您参考的是哪本教材或课程：

## 常见 LangChain 教材的章节安排

| 来源 | 可能的第三章内容 |
|------|--------------|
| **《LangChain 实战》** (李沐等) | 可能是 Chains 或 Agents |
| **LangChain 官方文档教程** | 可能是 Chains 或 Memory |
| **某在线课程/极客时间** | 需查看具体大纲 |
| **GitHub 开源教程** | 因项目而异 |

## 建议您提供以下信息

1. **书名/课程名** 是什么？
2. **作者或出版机构** 是谁？
3. 或者**第三章的标题**是什么？

---

如果您能告诉我第三章的**标题**或**开篇讲的核心概念**，我可以帮您：
- 解释该技术的原理
- 提供代码示例
- 说明其在 LangChain 中的位置

请补充信息，我可以给出更准确的解答。


### 刚才发生了什么？

模型要么编造了一个答案（幻觉），要么说不知道。因为 LLM 只了解训练数据中的内容，对你的私有文档、最新资料一无所知。

**RAG（Retrieval-Augmented Generation）** 就是为了解决这个问题：先从你的文档中检索相关内容，再把它作为上下文交给 LLM 生成回答。

```
用户问题 --> 检索相关文档 --> 将文档 + 问题一起发给 LLM --> 基于文档的准确回答

RAG = Retrieval（检索）+ Augmented（增强）+ Generation（生成）
```

核心思路：不是让模型什么都知道，而是在回答前给它「开卷考试」的材料。

## 2. Document 对象

在 RAG 流程中，所有文档都用 LangChain 的 `Document` 对象表示。先了解这个基础数据结构：

In [5]:
from langchain_core.documents import Document

doc = Document(
    page_content="LangChain 是一个用于构建 LLM 应用的框架。",
    metadata={"source": "intro.txt", "page": 1}
)

print(f"内容: {doc.page_content}")
print(f"元数据: {doc.metadata}")

内容: LangChain 是一个用于构建 LLM 应用的框架。
元数据: {'source': 'intro.txt', 'page': 1}


### 刚才发生了什么？

`Document` 只有两个属性：

| 属性 | 类型 | 说明 |
|------|------|------|
| `page_content` | `str` | 文档的文本内容 |
| `metadata` | `dict` | 来源、页码等附加信息 |

所有的 Loader 输出的都是 `Document` 列表，后续的分割、向量化、检索也都围绕这个对象展开。

## 3. 文档加载（Document Loaders）

先创建一份示例文档，内容就用前几章学过的知识点——这样 RAG 检索出来的东西你已经熟悉了：

In [6]:
# 创建示例文档
sample_text = """# LangChain 核心概念

## Models
LangChain 支持多种大语言模型的接入。通过统一的接口，你可以轻松切换不同的模型提供商。
核心接口是 ChatModel，它接收消息列表并返回 AI 消息。

## Prompt Templates
提示词模板让你可以复用和参数化提示词。ChatPromptTemplate 支持系统消息、用户消息和历史消息的组合。
MessagesPlaceholder 可以在模板中预留位置，用于插入动态的消息列表。

## LCEL
LangChain Expression Language 使用管道操作符 | 将组件串联成链。
每个组件都实现了 Runnable 接口，支持 invoke、stream、batch 三种调用方式。
RunnablePassthrough 用于透传输入，RunnableParallel 用于并行执行。

## Memory
对话记忆让模型能够"记住"之前的对话内容。
ChatMessageHistory 用于存储消息，RunnableWithMessageHistory 自动管理历史注入。
trim_messages 可以限制历史长度，避免超出上下文窗口。

## RAG
检索增强生成（RAG）让模型能够基于外部文档回答问题。
核心流程：加载文档 -> 分割 -> 向量化 -> 存储 -> 检索 -> 生成。
"""

with open("langchain_notes.txt", "w") as f:
    f.write(sample_text)

print("示例文档已创建")

示例文档已创建


In [7]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("langchain_notes.txt")
docs = loader.load()

print(f"加载了 {len(docs)} 个文档")
print(f"内容预览: {docs[0].page_content[:100]}...")
print(f"元数据: {docs[0].metadata}")

加载了 1 个文档
内容预览: # LangChain 核心概念

## Models
LangChain 支持多种大语言模型的接入。通过统一的接口，你可以轻松切换不同的模型提供商。
核心接口是 ChatModel，它接收消息列表并...
元数据: {'source': 'langchain_notes.txt'}


### 刚才发生了什么？

`TextLoader` 读取整个文件，返回包含一个 `Document` 的列表。元数据自动记录了文件路径。

### 常用 Document Loader

| Loader | 数据源 | 安装依赖 |
|--------|--------|----------|
| `TextLoader` | `.txt` 文本文件 | 无 |
| `PyPDFLoader` | PDF 文件 | `pypdf` |
| `CSVLoader` | CSV 文件 | 无 |
| `WebBaseLoader` | 网页 URL | `beautifulsoup4` |
| `DirectoryLoader` | 整个目录 | 无 |

### 常见问题

- **中文乱码**：`TextLoader("file.txt", encoding="utf-8")`，手动指定编码。
- **文件过大**：单个 Document 可能包含整本书的内容，需要后续的文本分割。

## 4. 文本分割（Text Splitters）

一个文档可能有几千甚至几万字。直接向量化和检索效果很差，需要切成小块。

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=50,
    separators=["\n\n", "\n", "。", "，", " ", ""]
)

chunks = splitter.split_documents(docs)

print(f"原始文档数: {len(docs)}")
print(f"分割后块数: {len(chunks)}")
print()
for i, chunk in enumerate(chunks):
    print(f"--- 块 {i} ({len(chunk.page_content)} 字符) ---")
    print(chunk.page_content[:80] + "...")
    print()

原始文档数: 1
分割后块数: 5

--- 块 0 (109 字符) ---
# LangChain 核心概念

## Models
LangChain 支持多种大语言模型的接入。通过统一的接口，你可以轻松切换不同的模型提供商。
核心接口...

--- 块 1 (122 字符) ---
## Prompt Templates
提示词模板让你可以复用和参数化提示词。ChatPromptTemplate 支持系统消息、用户消息和历史消息的组合。
M...

--- 块 2 (160 字符) ---
## LCEL
LangChain Expression Language 使用管道操作符 | 将组件串联成链。
每个组件都实现了 Runnable 接口，支持...

--- 块 3 (128 字符) ---
## Memory
对话记忆让模型能够"记住"之前的对话内容。
ChatMessageHistory 用于存储消息，RunnableWithMessageHis...

--- 块 4 (76 字符) ---
## RAG
检索增强生成（RAG）让模型能够基于外部文档回答问题。
核心流程：加载文档 -> 分割 -> 向量化 -> 存储 -> 检索 -> 生成。...



### 刚才发生了什么？

`RecursiveCharacterTextSplitter` 按照 `separators` 列表的优先级递归分割文本。它会先尝试用 `\n\n`（段落）拆分，拆不够小就用 `\n`（行），以此类推。

**chunk_overlap 的作用：**

```
文档:  [==========A==========][=====overlap=====][==========B==========]
块A:   [==========A==========][=====overlap=====]
块B:                          [=====overlap=====][==========B==========]
```

相邻块之间有一段重叠内容，确保被分割到边界处的语义不会丢失。

### 关键参数

| 参数 | 说明 | 建议值 |
|------|------|--------|
| `chunk_size` | 每块最大字符数 | 200-1000，取决于 embedding 模型和内容密度 |
| `chunk_overlap` | 相邻块重叠字符数 | chunk_size 的 10%-25% |
| `separators` | 分割符优先级列表 | 中文文档建议加上 `。`、`，` |

其他分割器：`CharacterTextSplitter`（单一分隔符）、`MarkdownTextSplitter`（按 Markdown 标题分割）。大多数情况 `RecursiveCharacterTextSplitter` 是最佳选择。

## 5. Embedding（文本向量化）

向量化是 RAG 的核心——把文本转成一组数字（向量），语义相近的文本在向量空间中距离更近。

In [14]:
# 单个文本向量化
vector = embeddings.embed_query("什么是 LangChain？")
print(f"向量维度: {len(vector)}")
print(f"前 5 个值: {vector[:5]}")

向量维度: 2048
前 5 个值: [-0.008681776, -0.01625694, -0.016222715, 0.017203836, -0.026421806]


In [20]:
# 多个文本向量化，计算相似度
texts = ["猫在睡觉", "猫在吃饭", "Python 编程"]
vectors = embeddings.embed_documents(texts)

import numpy as np

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print(f"'猫在睡觉' vs '猫在吃饭': {cosine_similarity(vectors[0], vectors[1]):.4f}")
print(f"'猫在睡觉' vs 'Python 编程': {cosine_similarity(vectors[0], vectors[2]):.4f}")

'猫在睡觉' vs '猫在吃饭': 0.7630
'猫在睡觉' vs 'Python 编程': 0.5736


### 刚才发生了什么？

- `embed_query()` 把单个查询文本转成向量
- `embed_documents()` 批量转换文档文本
- **余弦相似度（cosine similarity）**：值越接近 1，两段文本语义越相似

「猫在睡觉」和「狗在跑步」都是关于动物行为的描述，语义相似度高于与「Python 编程」的比较。

这就是向量检索的基础：把问题也转成向量，找到向量空间中最近的文档块。

### 常见问题

- **numpy 未安装**：`uv add numpy`，通常它作为传递依赖已经存在。
- **两个方法的区别**：`embed_query` 用于查询文本，`embed_documents` 用于文档文本。某些 embedding 模型对两者有不同的处理策略。

## 6. 向量存储（Vector Store）

有了向量，就需要一个地方存储它们并支持快速相似性搜索。FAISS 是 Meta 开源的向量检索库，适合本地使用。

In [21]:
from langchain_community.vectorstores import FAISS

# 一步完成：向量化所有文档块并存入 FAISS
vectorstore = FAISS.from_documents(chunks, embeddings)
print(f"向量库中有 {vectorstore.index.ntotal} 个向量")

向量库中有 5 个向量


In [22]:
# 相似性搜索
results = vectorstore.similarity_search("LCEL 是什么？", k=2)

for i, doc in enumerate(results):
    print(f"--- 结果 {i+1} ---")
    print(doc.page_content)
    print()

--- 结果 1 ---
## LCEL
LangChain Expression Language 使用管道操作符 | 将组件串联成链。
每个组件都实现了 Runnable 接口，支持 invoke、stream、batch 三种调用方式。
RunnablePassthrough 用于透传输入，RunnableParallel 用于并行执行。

--- 结果 2 ---
## Memory
对话记忆让模型能够"记住"之前的对话内容。
ChatMessageHistory 用于存储消息，RunnableWithMessageHistory 自动管理历史注入。
trim_messages 可以限制历史长度，避免超出上下文窗口。



In [23]:
# 带分数的搜索
results_with_scores = vectorstore.similarity_search_with_score("对话记忆", k=2)

for doc, score in results_with_scores:
    print(f"[距离分数: {score:.4f}]")
    print(doc.page_content)
    print()

[距离分数: 0.6583]
## Memory
对话记忆让模型能够"记住"之前的对话内容。
ChatMessageHistory 用于存储消息，RunnableWithMessageHistory 自动管理历史注入。
trim_messages 可以限制历史长度，避免超出上下文窗口。

[距离分数: 1.1250]
## Prompt Templates
提示词模板让你可以复用和参数化提示词。ChatPromptTemplate 支持系统消息、用户消息和历史消息的组合。
MessagesPlaceholder 可以在模板中预留位置，用于插入动态的消息列表。



### 刚才发生了什么？

`FAISS.from_documents()` 做了两件事：
1. 用 embedding 模型把每个文档块转成向量
2. 建立 FAISS 索引，支持快速最近邻搜索

`similarity_search(query, k)` 将查询文本向量化，然后在索引中找到最相似的 k 个文档块。

### 常见问题

- **分数含义**：FAISS 默认使用 L2 距离（欧氏距离），**分数越小越相似**，这与余弦相似度（越大越相似）相反，注意区分。
- **k 值选择**：k 太小可能遗漏相关信息，太大会引入噪声。通常 2-5 是合理范围。

## 7. Retriever（检索器接口）

Vector Store 提供了检索功能，但它不是 Runnable，无法直接用在 LCEL 链中。`as_retriever()` 将其包装为 Retriever——一个标准的 Runnable 组件。

In [30]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

# Retriever 是 Runnable，可以用 invoke
docs = retriever.invoke("如何管理对话历史？")

for doc in docs:
    print(doc.metadata)
    print(doc.page_content)
    print("---")

{'source': 'langchain_notes.txt'}
## Memory
对话记忆让模型能够"记住"之前的对话内容。
ChatMessageHistory 用于存储消息，RunnableWithMessageHistory 自动管理历史注入。
trim_messages 可以限制历史长度，避免超出上下文窗口。
---
{'source': 'langchain_notes.txt'}
## RAG
检索增强生成（RAG）让模型能够基于外部文档回答问题。
核心流程：加载文档 -> 分割 -> 向量化 -> 存储 -> 检索 -> 生成。
---


### 刚才发生了什么？

`as_retriever()` 返回一个 `VectorStoreRetriever`，它实现了 Runnable 接口：

- 输入：`str`（查询文本）
- 输出：`list[Document]`（检索到的文档列表）

因为是 Runnable，所以可以直接用 `|` 接入 LCEL 链——这正是下一节要做的事。

## 8. 构建 RAG 链

万事俱备。把 Retriever + Prompt + LLM 用 LCEL 组合成完整的 RAG 链：

In [25]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

def format_docs(docs):
    """将 Document 列表拼接为纯文本"""
    return "\n\n".join(doc.page_content for doc in docs)

rag_prompt = ChatPromptTemplate.from_messages([
    ("system", "基于以下参考资料回答用户问题。如果资料中没有相关信息，请说明。\n\n参考资料：\n{context}"),
    ("human", "{question}"),
])

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

result = rag_chain.invoke("LCEL 的管道操作符有什么用？")
print(result)

根据参考资料，LCEL 的管道操作符 **`|`** 用于**将组件串联成链**。

具体来说：

- 它可以把多个实现了 `Runnable` 接口的组件连接起来
- 形成一条可执行的调用链（chain）

这种设计让每个组件都能保持独立，同时通过管道操作符灵活组合，支持 `invoke`、`stream`、`batch` 三种调用方式。


In [26]:
# 再问一个问题
result = rag_chain.invoke("如何控制对话历史的长度？")
print(result)

根据参考资料，控制对话历史长度可以使用 **trim_messages** 功能：

> **trim_messages** 可以限制历史长度，避免超出上下文窗口。

## 具体说明

`trim_messages` 的作用：
- **限制历史长度** — 防止对话历史过长导致超出模型的上下文窗口限制
- **自动截断** — 保留最近的消息，移除较早的消息

## 使用场景

通常在以下场景需要控制历史长度：
1. 长对话场景，历史消息可能累积很多
2. 模型上下文窗口有限（如 GPT-3.5 的 4K/16K token 限制）
3. 需要控制成本和响应速度

## 配合使用的组件

```python
# 典型组合
from langchain.memory import trim_messages
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

# trim_messages 通常与这些组件一起使用：
# - ChatMessageHistory: 存储消息
# - RunnableWithMessageHistory: 自动管理历史注入
# - MessagesPlaceholder: 在模板中预留动态消息位置
```

---

**注意**：参考资料中只提到了 `trim_messages` 这个核心功能，但没有提供具体的参数设置（如按 token 数截断还是按消息数截断）和代码示例。如需详细的实现方式，建议查阅 LangChain 官方文档。


### 刚才发生了什么？

这是 RAG 的核心模式。数据流如下：

```
问题 --+-- retriever -> format_docs -> context --+
       |                                         |
       +-- RunnablePassthrough -> question -------+-> rag_prompt -> llm -> StrOutputParser -> 回答
```

拆解每一步：

1. 用户输入一个字符串（问题）
2. **并行执行两条路径**（字典 = `RunnableParallel`）：
   - `retriever | format_docs`：用问题检索相关文档，拼接成文本
   - `RunnablePassthrough()`：原样透传问题
3. 结果合并为 `{"context": "...", "question": "..."}` 字典
4. 字典填入 `rag_prompt` 模板
5. 发给 LLM 生成回答
6. `StrOutputParser` 提取纯文本

注意：第 3 章学的 `RunnablePassthrough` 和字典语法在这里派上了用场。RAG 链是 LCEL 最经典的应用场景。

## 9. RAG 链进阶

### 追踪引用来源

用户可能想知道回答是基于哪些文档块生成的。用 `RunnableParallel` 同时返回回答和来源文档：

In [27]:
from langchain_core.runnables import RunnableParallel

rag_with_sources = RunnableParallel(
    answer=rag_chain,
    sources=retriever,
)

result = rag_with_sources.invoke("什么是 Prompt Templates？")
print("回答:", result["answer"])
print("\n引用来源:")
for doc in result["sources"]:
    print(f"  - {doc.metadata}")

回答: 根据参考资料，**Prompt Templates（提示词模板）** 是一种让你可以**复用和参数化提示词**的工具。

具体来说：

- **ChatPromptTemplate** 支持将系统消息、用户消息和历史消息组合在一起，形成结构化的提示词
- **MessagesPlaceholder** 可以在模板中预留位置，用于插入动态的消息列表（比如对话历史）

简单来说，提示词模板的作用就是让你不用每次都手写完整的提示词，而是定义一个可复用的"模板"，在需要时填入不同的变量或内容即可。

引用来源:
  - {'source': 'langchain_notes.txt'}
  - {'source': 'langchain_notes.txt'}


In [29]:
# RAG 链也支持流式输出
for chunk in rag_chain.stream("RunnablePassthrough 有什么作用？"):
    print(chunk, end="", flush=True)
print()

根据提供的参考资料，其中**没有关于 RunnablePassthrough 的相关信息**。

参考资料主要介绍了：
- **Prompt Templates（提示词模板）**：包括 ChatPromptTemplate 和 MessagesPlaceholder
- **Memory（对话记忆）**：包括 ChatMessageHistory、RunnableWithMessageHistory 和 trim_messages

如需了解 RunnablePassthrough 的作用，建议查阅其他相关文档或资料。


### 刚才发生了什么？

- `RunnableParallel` 让 `rag_chain`（回答）和 `retriever`（来源文档）并行执行，结果合并为字典。
- 流式输出同样开箱即用——因为 `rag_chain` 中每个组件都是 Runnable，LCEL 自动透传流式 token。

在生产环境中，引用来源是必备功能——让用户能验证回答的可信度。

## 10. 向量库的保存与加载

每次都重新向量化文档很慢（尤其文档量大时）。FAISS 支持将索引保存到本地磁盘：

In [31]:
# 保存到本地
vectorstore.save_local("faiss_index")
print("向量库已保存到 faiss_index/")

向量库已保存到 faiss_index/


In [33]:
# 从本地加载
loaded_vectorstore = FAISS.load_local(
    "faiss_index",
    embeddings,
    allow_dangerous_deserialization=True  # FAISS 内部使用 pickle
)

# 验证加载成功
results = loaded_vectorstore.similarity_search("LangChain", k=1)
print(f"加载成功，检索到: {results[0].page_content[:50]}...")

加载成功，检索到: ## LCEL
LangChain Expression Language 使用管道操作符 | 将组...


### 刚才发生了什么？

- `save_local(path)` 将 FAISS 索引和文档元数据序列化到指定目录
- `load_local(path, embeddings)` 从磁盘恢复，需要传入同一个 embedding 模型（后续查询要用它向量化新的问题）

### 常见问题

- **`allow_dangerous_deserialization=True`**：FAISS 的 `load_local` 内部使用 pickle 反序列化，存在代码执行风险。LangChain 要求你显式确认知道这一点。只加载自己创建的索引时可以安全使用。
- **索引与 embedding 模型绑定**：用模型 A 创建的索引不能用模型 B 加载——向量维度和语义空间不同。

## 11. 实战：基于本地文档的问答系统

把前面学到的所有步骤封装成一个可复用的函数：

In [34]:
def build_rag_system(file_path, chunk_size=300, chunk_overlap=50):
    """从文件构建 RAG 问答系统"""
    # 1. 加载
    loader = TextLoader(file_path, encoding="utf-8")
    docs = loader.load()

    # 2. 分割
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
    )
    chunks = splitter.split_documents(docs)

    # 3. 向量化 + 存储
    vectorstore = FAISS.from_documents(chunks, embeddings)
    retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

    # 4. 构建链
    prompt = ChatPromptTemplate.from_messages([
        ("system", "你是一个知识助手。基于以下参考资料回答问题，引用相关内容。"
                   "如果资料中没有相关信息，明确告知用户。\n\n参考资料：\n{context}"),
        ("human", "{question}"),
    ])

    chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )

    return chain

print("build_rag_system 函数定义完成")

build_rag_system 函数定义完成


In [35]:
# 使用
qa = build_rag_system("langchain_notes.txt")

questions = [
    "LangChain 支持哪些调用方式？",
    "什么是 MessagesPlaceholder？",
    "RAG 的核心流程是什么？",
]

for q in questions:
    print(f"问: {q}")
    print(f"答: {qa.invoke(q)}")
    print("-" * 50)

问: LangChain 支持哪些调用方式？
答: 根据参考资料，LangChain 中的每个组件都实现了 **Runnable** 接口，支持以下三种调用方式：

| 调用方式 | 说明 |
|---------|------|
| **invoke** | 单个输入的同步调用 |
| **stream** | 流式输出，逐步返回结果 |
| **batch** | 批量处理多个输入 |

这些调用方式是 LCEL（LangChain Expression Language）的核心特性，通过管道操作符 `|` 将组件串联成链后，可以使用上述任意方式执行。
--------------------------------------------------
问: 什么是 MessagesPlaceholder？
答: 根据参考资料，**MessagesPlaceholder** 是提示词模板（Prompt Templates）中的一个重要组件。

## 定义与作用

MessagesPlaceholder 可以在模板中**预留位置**，用于**插入动态的消息列表**。

## 典型使用场景

它通常用于在 ChatPromptTemplate 中预留对话历史的位置，让历史消息能够动态注入到提示词中。例如：

```python
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个有帮助的助手。"),
    MessagesPlaceholder(variable_name="history"),  # 预留历史消息位置
    ("human", "{input}")
])
```

## 与其他组件的配合

- 常与 **ChatPromptTemplate** 配合使用，构建包含系统消息、用户消息和历史消息的完整提示
- 可与 **RunnableWithMessageHistory** 结合，实现对话历史的自动管理和注入

这样设计的好处是将**静态的模板结构**与**动态的消息内容**解耦，使提示词模板更加灵活可复用。
-----------

### 刚才发生了什么？

`build_rag_system()` 封装了完整的 RAG 流程：

```
文件路径 -> TextLoader -> Documents -> Splitter -> Chunks -> FAISS -> Retriever -> RAG Chain
```

返回的 `chain` 是一个 Runnable，支持 `invoke`、`stream`、`batch`。只需传入文件路径，即可得到一个基于该文件内容的问答系统。

对比第 1 节：现在模型能准确回答关于我们笔记内容的问题了——这就是 RAG 的价值。

## 12. 清理临时文件

In [36]:
import shutil

if os.path.exists("langchain_notes.txt"):
    os.remove("langchain_notes.txt")
if os.path.exists("faiss_index"):
    shutil.rmtree("faiss_index")
print("临时文件已清理")

临时文件已清理


## 13. 总结

本章学习了：

- 理解了 RAG 解决什么问题：让 LLM 基于私有/最新文档回答问题
- `Document` 对象：`page_content` + `metadata`
- 文档加载：`TextLoader` 等各种 Loader
- 文本分割：`RecursiveCharacterTextSplitter`，控制 `chunk_size` 和 `chunk_overlap`
- Embedding 向量化：文本转向量，余弦相似度衡量语义距离
- 向量存储与检索：FAISS，`similarity_search`
- Retriever 接口：`as_retriever()` 将向量库变为 Runnable
- 构建 RAG 链：`retriever | format_docs` + LCEL
- 进阶：引用来源追踪、流式输出、向量库持久化

### 速查表

| 组件 | 作用 | 关键类/方法 |
|------|------|-------------|
| Document | 文档抽象 | `Document(page_content, metadata)` |
| Loader | 加载文档 | `TextLoader`, `PyPDFLoader` |
| Splitter | 分割文档 | `RecursiveCharacterTextSplitter` |
| Embedding | 文本转向量 | `OpenAIEmbeddings` |
| VectorStore | 向量存储与检索 | `FAISS` |
| Retriever | 检索器（Runnable） | `vectorstore.as_retriever()` |
| RAG Chain | 检索 + 生成 | `retriever \| format_docs` + `prompt \| llm` |

### RAG 完整流程

```
原始文档 -> Loader -> Documents -> Splitter -> Chunks -> Embedding -> VectorStore
                                                                         |
用户问题 -> Retriever -> 相关文档 -> Prompt（问题+文档） -> LLM -> 回答
```

## 下一章预告

**第六章：Agents（工具调用与自主决策）** —— 到目前为止，链的执行路径都是预先定义好的。下一章将学习如何让模型自主决定使用哪些工具、以什么顺序执行，实现真正的「智能体」。